# 01 — Data Overview

Raw data inspection — no modifications, read only.
At the end, write a decision note on which columns to keep.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from Source.scripts.load_data import DATASET_PATHS, load_named_dataset

print('Project root:', PROJECT_ROOT)
print('Available datasets:', list(DATASET_PATHS.keys()))

## 1. movies_metadata

Primary source — budget, revenue, genre, release_date and imdb_id are here.

In [ ]:
meta = load_named_dataset('the_movies_metadata')
print('Shape:', meta.shape)
meta.head(3)

In [ ]:
meta.info()

In [ ]:
meta['budget'] = pd.to_numeric(meta['budget'], errors='coerce')
meta['revenue'] = pd.to_numeric(meta['revenue'], errors='coerce')

usable = ((meta['budget'] > 0) & (meta['revenue'] > 0)).sum()
print(f'Rows with budget > 0 AND revenue > 0: {usable} / {len(meta)}')
print(f'Budget nulls : {meta["budget"].isna().sum()}')
print(f'Revenue nulls: {meta["revenue"].isna().sum()}')
meta[['budget', 'revenue', 'vote_average', 'popularity']].describe()

In [ ]:
# genres column is a JSON string — inspect its format
meta['genres'].dropna().head(5).tolist()

## 2. tmdb_5000_movies

TMDB's own dataset — for popularity and additional budget/revenue validation.

In [ ]:
tmdb = load_named_dataset('tmdb_movies')
print('Shape:', tmdb.shape)
tmdb.head(3)

In [ ]:
usable_tmdb = ((tmdb['budget'] > 0) & (tmdb['revenue'] > 0)).sum()
print(f'Rows with budget > 0 AND revenue > 0: {usable_tmdb} / {len(tmdb)}')
tmdb[['budget', 'revenue', 'vote_average', 'popularity']].describe()

## 3. Rotten Tomatoes Movies

For tomatometer (critic score) and audience rating.

In [ ]:
rt = load_named_dataset('rt_movies')
print('Shape:', rt.shape)
rt.head(3)

In [ ]:
rating_cols = [c for c in rt.columns if 'rating' in c.lower() or 'score' in c.lower()]
print('Rating-related columns:', rating_cols)
print('\nNull counts:')
print(rt[rating_cols].isna().sum())
print('\nTitle column:', [c for c in rt.columns if 'title' in c.lower()])
rt[rating_cols].describe()

## 4. IMDb Ratings

Most reliable rating source — averageRating and numVotes.
Will be joined via imdb_id, which is much more reliable than title matching.

In [ ]:
imdb = load_named_dataset('imdb_ratings')
print('Shape:', imdb.shape)
imdb.head(5)

In [ ]:
imdb[['averageRating', 'numVotes']].describe()

In [ ]:
# Does movies_metadata have imdb_id? Critical for the join.
print('imdb_id in movies_metadata:', 'imdb_id' in meta.columns)
print('Sample imdb_id values:', meta['imdb_id'].dropna().head(5).tolist())
print('Sample tconst values :', imdb['tconst'].head(5).tolist())

## Decision Note

Copied to `Documentation/reports/data_overview.md`.

### Columns to use

| Source | Columns | Purpose |
|---|---|---|
| movies_metadata | budget, revenue, title, imdb_id, genres, release_date, runtime, vote_average | Primary source |
| tmdb_movies | popularity | Additional feature (as tmdb_popularity) |
| imdb_ratings | averageRating, numVotes | Rating source (joined via imdb_id) |
| rt_movies | tomatometer_rating, audience_rating | Secondary rating source (joined via movie_title) |

### Observations

- movies_metadata total rows: 45,466 — usable (budget > 0 AND revenue > 0): **5,381**
- tmdb_movies total rows: 4,803 — usable (budget > 0 AND revenue > 0): **3,229**
- IMDb ratings total rows: 1,666,284
- RT movies total rows: 17,712
- imdb_id format match: **Yes** — both use `tt` prefix (e.g. `tt0114709` vs `tt0000001`)
- RT tomatometer_rating null rate: **0.2%** (very complete)
- RT audience_rating null rate: **1.7%**
- RT title column name: `movie_title` (not `title`) — handled in merge step
- Primary merge base: movies_metadata (5,381 rows after filter)